# Кейс №2 — экономика инференса GLM-5.2 в Sail Research

Воспроизведение финального слайда. Дата сборки и проверки: **14.09.2026**. Исходный ценовой сценарий слайда — **13.09.2026**, а не подтверждённый действующий тариф.

Считаем **GPU contribution margin**, не бухгалтерскую маржу Sail. Production-конфигурация Sail не раскрывается; 8×H200 — расчётная конфигурация по публичному benchmark. Полный реестр источников и статус проверки — в [README](../README.md).

In [1]:
from pathlib import Path
import sys, json, math, subprocess
from IPython.display import display, Markdown

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src/case_2_sail_glm52_economics.py").is_file())
sys.path.insert(0, str(ROOT / "src"))
from case_2_sail_glm52_economics import load_assumptions, calculate, sensitivity, check_slide
a = load_assumptions(ROOT / "data/assumptions.csv")
a

{'input_price_usd_mtok': 0.5,
 'output_price_usd_mtok': 3.15,
 'input_tokens': 8192.0,
 'output_tokens': 1024.0,
 'total_throughput_tok_s': 17992.6,
 'gpu_count': 8.0,
 'gpu_price_usd_h': 2.63,
 'utilization': 1.0,
 'sensitivity_low': 1.98,
 'sensitivity_high': 5.86,
 'sensitivity_rounded_break_even': 6.43}

## 1. Тариф и выручка запроса

Выручка запроса = `(8192 × 0.50 + 1024 × 3.15) / 1 000 000`. Вход без кеш-скидки; все 1024 выходных токена оплачиваются по выходному тарифу.

In [2]:
request_revenue = (a["input_tokens"] * a["input_price_usd_mtok"]
                   + a["output_tokens"] * a["output_price_usd_mtok"]) / 1_000_000
print(f"Выручка запроса: ${request_revenue:.7f}")

Выручка запроса: $0.0073216


## 2. Производительность и часовая выручка

17 992,6 — **total input + output tok/s на весь узел**. Делим на 9216 токенов запроса. Базовый сценарий предполагает непрерывную продажу 100% benchmark-throughput; это не измеренная загрузка Sail.

In [3]:
requests_s = a["total_throughput_tok_s"] / (a["input_tokens"] + a["output_tokens"])
revenue_h = requests_s * 3600 * request_revenue * a["utilization"]
print(f"Запросов/с: {requests_s:.6f}; запросов/ч: {requests_s * 3600:.6f}")
print(f"Выручка узла: ${revenue_h:.6f}/ч → ≈${revenue_h:.1f}/ч")

Запросов/с: 1.952322; запросов/ч: 7028.359375
Выручка узла: $51.458836/ч → ≈$51.5/ч


## 3. Стоимость GPU, contribution margin и break-even

`Стоимость = 8 × ставка GPU`; `contribution = выручка − стоимость`; `маржа = contribution / выручка`; `break-even GPU = выручка / 8`. Промежуточные значения не округляем.

In [4]:
cost_h = a["gpu_count"] * a["gpu_price_usd_h"]
margin_pct = 100 * (revenue_h - cost_h) / revenue_h
break_even = revenue_h / a["gpu_count"]
print(f"GPU-cost: ${cost_h:.2f}/ч → ≈${cost_h:.1f}/ч")
print(f"GPU contribution: ${revenue_h - cost_h:.6f}/ч")
print(f"GPU contribution margin: {margin_pct:.6f}% → ≈{margin_pct:.0f}%")
print(f"Break-even: ${break_even:.7f}/GPU·ч → ≈${break_even:.2f}/GPU·ч")

GPU-cost: $21.04/ч → ≈$21.0/ч
GPU contribution: $30.418836/ч
GPU contribution margin: 59.112950% → ≈59%
Break-even: $6.4323545/GPU·ч → ≈$6.43/GPU·ч


## 4. Чувствительность к цене H200

Меняется только ставка аренды, throughput и профиль запросов фиксированы. Рыночные границы — сценарии, а не взаимозаменяемые предложения узла с одинаковым SLA.

In [5]:
scenarios = sensitivity(a)
table = "| $/GPU·ч | GPU-cost $/ч | Маржа, % | На слайде |\n|---:|---:|---:|---:|\n"
for row in scenarios:
    table += f"| {row['gpu_price_usd_h']:.2f} | {row['gpu_cost_node_usd_h']:.2f} | {row['gpu_margin_pct']:.6f} | ≈{row['gpu_margin_pct']:.0f}% |\n"
display(Markdown(table))
print(f"При $6.43 маржа {scenarios[-1]['gpu_margin_pct']:.6f}%, а не точный ноль.")

| $/GPU·ч | GPU-cost $/ч | Маржа, % | На слайде |
|---:|---:|---:|---:|
| 1.98 | 15.84 | 69.218114 | ≈69% |
| 2.63 | 21.04 | 59.112950 | ≈59% |
| 5.86 | 46.88 | 8.898056 | ≈9% |
| 6.43 | 51.44 | 0.036604 | ≈0% |


При $6.43 маржа 0.036604%, а не точный ноль.


## 5. Проверка относительно слайда и исполняемого скрипта

Сверяем округлённые контрольные значения, расчёт по шагам и JSON отдельного процесса. Точный break-even должен давать нулевую маржу.

In [6]:
check_slide(a)
r = calculate(a)
for key, value in {"request_revenue_usd": request_revenue, "requests_s": requests_s,
                   "revenue_node_usd_h": revenue_h, "gpu_cost_node_usd_h": cost_h,
                   "gpu_margin_pct": margin_pct, "break_even_gpu_usd_h": break_even}.items():
    assert math.isclose(r[key], value, rel_tol=1e-12, abs_tol=1e-12), key
script = subprocess.run([sys.executable, str(ROOT / "src/case_2_sail_glm52_economics.py"),
                         "--check-slide", "--json"], check=True, capture_output=True, text=True)
assert json.loads(script.stdout) == {"base": r, "sensitivity": scenarios}
assert abs(calculate(a, gpu_price=break_even)["gpu_margin_pct"]) < 1e-10
assert calculate(a, gpu_price=break_even + 0.01)["gpu_margin_pct"] < 0
print("PASS: Notebook = script; все KPI и 4 сценария совпадают со слайдом после округления.")

PASS: Notebook = script; все KPI и 4 сценария совпадают со слайдом после округления.


## 6. Ограничения и загрузка

Синтетический benchmark не гарантирует production-throughput или SLA. Подробности выбранного режима и ограничений источника приведены в README. Не добавляем отдельные CPU, сеть, хранение, резерв, персонал и прочие расходы; арендная цена может включать часть инфраструктуры. Не моделируем скидки, кеширование, бесплатные кредиты и комиссии.

Дополнительно проверяем снижение реализованной производительности: стоимость аренды остаётся полной. Это анализ допущения, а не оценка фактической загрузки Sail.

In [7]:
table = "| Доля benchmark-throughput | Выручка $/ч | GPU-маржа |\n|---:|---:|---:|\n"
for u in (1.0, 0.75, 0.5, 0.4):
    row = calculate(a, utilization=u)
    table += f"| {u:.0%} | {row['revenue_node_usd_h']:.2f} | {row['gpu_margin_pct']:.2f}% |\n"
display(Markdown(table))
print(f"Безубыточность по доле throughput при $2.63: {cost_h / revenue_h:.2%}")

| Доля benchmark-throughput | Выручка $/ч | GPU-маржа |
|---:|---:|---:|
| 100% | 51.46 | 59.11% |
| 75% | 38.59 | 45.48% |
| 50% | 25.73 | 18.23% |
| 40% | 20.58 | -2.22% |


Безубыточность по доле throughput при $2.63: 40.89%
